## 0. 환경설정

In [54]:
!pip install neo4j-graphrag neo4j openai mysql-connector-python pandas gradio python-dotenv -q


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [55]:
import os
import json
from dotenv import load_dotenv

load_dotenv()  # .env 파일에서 환경변수 로드

import pandas as pd
import mysql.connector
import gradio as gr

from neo4j import GraphDatabase, basic_auth
from neo4j.time import Date

from neo4j_graphrag.retrievers import Text2CypherRetriever
try:
    from neo4j_graphrag.llm import OpenAILLM
except ImportError:
    from neo4j_graphrag.llm.openai_llm import OpenAILLM
from neo4j_graphrag.generation import GraphRAG
import re
import time

import requests
from math import radians, cos, sin, asin, sqrt, atan2

print("모듈 로드 완료")

모듈 로드 완료


In [56]:
# OPENAI_API_KEY는 .env에서 자동 로드됨
# 확인용:
print("OpenAI API Key 설정됨:", bool(os.environ.get("OPENAI_API_KEY")))

OpenAI API Key 설정됨: True


In [57]:
# KAKAO_REST_API_KEY는 .env에서 자동 로드됨
# 확인용:
print("Kakao API Key 설정됨:", bool(os.environ.get("KAKAO_REST_API_KEY")))

Kakao API Key 설정됨: True


In [58]:
api_key  = os.environ.get("KAKAO_REST_API_KEY")
print("exists:", bool(api_key))
print("repr:", repr(api_key))
print("prefix:", api_key[:8] if api_key else None)

exists: True
repr: 'aa921e29bd619d0b1ad79ece4078f34b'
prefix: aa921e29


## 0-1. MY SQL 및 NEO4J 불러오기

In [59]:
MYSQL_HOST = os.environ.get("MYSQL_HOST", "localhost")
MYSQL_USER = os.environ.get("MYSQL_USER", "")
MYSQL_PASSWORD = os.environ.get("MYSQL_PASSWORD", "")
MYSQL_DB = os.environ.get("MYSQL_DB", "")
MYSQL_PORT = int(os.environ.get("MYSQL_PORT", 3306))

In [60]:
NEO4J_URI = os.environ.get("NEO4J_URI", "neo4j://127.0.0.1:7687")
NEO4J_USER = os.environ.get("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.environ.get("NEO4J_PW", "")

In [61]:
COUPLE_ID = 14  # 실제 존재하는 커플 ID


In [62]:
conn = None
if all([MYSQL_HOST, MYSQL_USER, MYSQL_PASSWORD, MYSQL_DB]):
    try:
        conn = mysql.connector.connect(
            host=MYSQL_HOST,
            user=MYSQL_USER,
            password=MYSQL_PASSWORD,
            database=MYSQL_DB,
            port=MYSQL_PORT
        )
        print("MySQL 연결 성공")
    except Exception as e:
        conn = None
        print(f"MySQL 연결 실패: {e}")
else:
    print("MySQL 미설정 (.env 확인 필요)")

MySQL 연결 성공


### 0-1-1. 사용자 선호 조회

In [63]:
query = """
SELECT *
FROM COUPLE_PREFERENCES
WHERE couple_id = %s
"""

df_pref = pd.read_sql(query, conn, params=(COUPLE_ID,))

if df_pref.empty:
    print(f"couple_id={COUPLE_ID} 선호 데이터 없음 (더미 사용)")
    pref = {"couple_id": COUPLE_ID, "region": "서울", "budget": "300만원"}
else:
    pref = df_pref.iloc[0].to_dict()
    print("=== 사용자 선호 ===")
    print(pref)
    print("=== 컬럼 목록 ===")
    print(df_pref.columns.tolist())


couple_id=2 선호 데이터 없음 (더미 사용)


C:\Users\SSAFY\AppData\Local\Temp\ipykernel_26648\3896295995.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_pref = pd.read_sql(query, conn, params=(COUPLE_ID,))


### 0-1-2. 사용자 취향

In [64]:
def get_user_preference(conn, couple_id):
    query = """
    SELECT *
    FROM couple_preferences
    WHERE couple_id = %s
    """
    df = pd.read_sql(query, conn, params=(couple_id,))

    if df.empty:
        return None

    return df.iloc[0].to_dict()


### 0-1-3. 좋아요

In [65]:
def get_user_likes(conn, couple_id):
    # couple_venue_likes 테이블이 없으므로 빈 리스트 반환
    return []


### 0-1-4. 좋아요 누른 홀이름

In [66]:
def get_liked_halls_with_names(conn, couple_id):
    # couple_venue_likes 테이블이 없으므로 빈 리스트 반환
    return []


### 0-1-5. 좋아요 기반 추천(+ 평점 순)

In [67]:
def recommend_by_likes(conn, couple_id):
    """
    couple_venue_likes 테이블이 없으므로,
    couple_preferences 기반으로 vendor_hall_detail 매칭 추천 반환.
    """
    pref = get_user_preference(conn, couple_id)
    if not pref:
        return []

    style = pref.get("style") or ""
    guest_count_raw = pref.get("guest_count") or ""
    budget_raw = pref.get("budget") or ""

    # guest_count 파싱 (예: "200명", "100~150명")
    import re
    guest_nums = [int(x) for x in re.findall(r"\d+", guest_count_raw)]
    guest_min = guest_nums[0] if guest_nums else 0
    guest_max = guest_nums[-1] if len(guest_nums) > 1 else guest_min

    # budget 파싱 (만원 단위 → 원)
    budget_nums = [int(x) for x in re.findall(r"\d+", budget_raw)]
    budget_val = budget_nums[0] * 10000 if budget_nums else None

    query = """
    SELECT
        v.id AS vendor_id,
        v.name,
        v.rating,
        v.review_count AS reviewCnt,
        vhd.style,
        vhd.guest_min,
        vhd.guest_max,
        vhd.meal_type,
        vhd.has_subway,
        vhd.has_parking,
        vp.price
    FROM vendor v
    JOIN vendor_hall_detail vhd ON v.id = vhd.vendor_id
    LEFT JOIN vendor_package vp ON v.id = vp.vendor_id
    WHERE v.category = '웨딩홀'
      AND (%s = '' OR vhd.style = %s)
      AND (%s = 0 OR vhd.guest_min <= %s)
      AND (%s = 0 OR vhd.guest_max >= %s)
    GROUP BY v.id, v.name, v.rating, v.review_count,
             vhd.style, vhd.guest_min, vhd.guest_max, vhd.meal_type,
             vhd.has_subway, vhd.has_parking, vp.price
    ORDER BY v.rating DESC, v.review_count DESC
    LIMIT 5
    """

    params = (style, style, guest_min, guest_min, guest_max if guest_max else 9999, guest_max if guest_max else 9999)
    df = pd.read_sql(query, conn, params=params)
    return df.to_dict(orient="records")


In [68]:
def recommend_by_likes_with_region_boost(conn, couple_id, preferred_region=None, preferred_sub_region=None):
    """
    couple_preferences + vendor + vendor_hall_detail 기반 추천.
    vendor 테이블에는 region 컬럼이 없으므로 description/hashtags로 지역 필터.
    """
    pref = get_user_preference(conn, couple_id)
    if not pref:
        return []

    style = pref.get("style") or ""
    guest_count_raw = pref.get("guest_count") or ""

    import re
    guest_nums = [int(x) for x in re.findall(r"\d+", guest_count_raw)]
    guest_min = guest_nums[0] if guest_nums else 0
    guest_max = guest_nums[-1] if len(guest_nums) > 1 else (guest_min if guest_min else 9999)

    region_filter = ""
    params = [style, style, guest_min, guest_min, guest_max, guest_max]

    if preferred_region:
        region_filter = "AND (v.description LIKE %s OR v.hashtags LIKE %s)"
        params += [f"%{preferred_region}%", f"%{preferred_region}%"]

    query = f"""
    SELECT
        v.id AS vendor_id,
        v.name,
        v.rating,
        v.review_count AS reviewCnt,
        vhd.style,
        vhd.guest_min,
        vhd.guest_max,
        vhd.meal_type,
        vhd.has_subway,
        vhd.has_parking,
        vp.price
    FROM vendor v
    JOIN vendor_hall_detail vhd ON v.id = vhd.vendor_id
    LEFT JOIN vendor_package vp ON v.id = vp.vendor_id
    WHERE v.category = '웨딩홀'
      AND (%s = '' OR vhd.style = %s)
      AND (%s = 0 OR vhd.guest_min <= %s)
      AND (%s = 0 OR vhd.guest_max >= %s)
      {region_filter}
    GROUP BY v.id, v.name, v.rating, v.review_count,
             vhd.style, vhd.guest_min, vhd.guest_max, vhd.meal_type,
             vhd.has_subway, vhd.has_parking, vp.price
    ORDER BY v.rating DESC, v.review_count DESC
    LIMIT 5
    """

    df = pd.read_sql(query, conn, params=params)
    return df.to_dict(orient="records")


## NEO4J 연결

In [69]:
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=basic_auth(NEO4J_USER, NEO4J_PASSWORD)
)

In [70]:
with driver.session() as session:
    count_result = session.run("MATCH (n) RETURN count(n) AS cnt")
    print("Neo4j node count:", count_result.single()["cnt"])

Neo4j node count: 4667


## 스키마 함수 추출

In [71]:
def get_node_datatype(value):
    if isinstance(value, str):
        return "STRING"
    elif isinstance(value, int):
        return "INTEGER"
    elif isinstance(value, float):
        return "FLOAT"
    elif isinstance(value, bool):
        return "BOOLEAN"
    elif isinstance(value, list):
        return f"LIST[{get_node_datatype(value[0])}]" if value else "LIST"
    elif isinstance(value, Date):
        return "DATE"
    else:
        return "UNKNOWN"

In [72]:
def get_schema(uri, user, password):
    driver_local = GraphDatabase.driver(uri, auth=basic_auth(user, password))

    with driver_local.session() as session:
        node_query = """
        MATCH (n)
        WITH DISTINCT labels(n) AS node_labels, keys(n) AS property_keys, n
        UNWIND node_labels AS label
        UNWIND property_keys AS key
        RETURN label, key, n[key] AS sample_value
        """

        rel_query = """
        MATCH ()-[r]->()
        WITH DISTINCT type(r) AS rel_type, keys(r) AS property_keys, r
        UNWIND property_keys AS key
        RETURN rel_type, key, r[key] AS sample_value
        """

        rel_direction_query = """
        MATCH (a)-[r]->(b)
        RETURN DISTINCT labels(a) AS start_label, type(r) AS rel_type, labels(b) AS end_label
        ORDER BY start_label, rel_type, end_label
        """

        nodes = session.run(node_query)
        relationships = session.run(rel_query)
        rel_directions = session.run(rel_direction_query)

        schema = {
            "nodes": {},
            "relationships": {},
            "relations": []
        }

        for record in nodes:
            label = record["label"]
            key = record["key"]
            sample_value = record["sample_value"]
            inferred_type = get_node_datatype(sample_value)

            if label not in schema["nodes"]:
                schema["nodes"][label] = {}
            schema["nodes"][label][key] = inferred_type

        for record in relationships:
            rel_type = record["rel_type"]
            key = record["key"]
            sample_value = record["sample_value"]
            inferred_type = get_node_datatype(sample_value)

            if rel_type not in schema["relationships"]:
                schema["relationships"][rel_type] = {}
            schema["relationships"][rel_type][key] = inferred_type

        for record in rel_directions:
            start_labels = record["start_label"]
            end_labels = record["end_label"]

            if not start_labels or not end_labels:
                continue

            start_label = start_labels[0]
            rel_type = record["rel_type"]
            end_label = end_labels[0]

            schema["relations"].append(f"(:{start_label})-[:{rel_type}]->(:{end_label})")

    driver_local.close()
    return schema

In [73]:
def format_schema(schema):
    result = []
    result.append("Node properties:")
    for label, properties in schema["nodes"].items():
        props = ", ".join(f"{k}: {v}" for k, v in properties.items())
        result.append(f"{label} {{{props}}}")
    result.append("Relationship properties:")
    for rel_type, properties in schema["relationships"].items():
        props = ", ".join(f"{k}: {v}" for k, v in properties.items())
        result.append(f"{rel_type} {{{props}}}")
    result.append("The relationships:")
    for relation in schema["relations"]:
        result.append(relation)
    return "\n".join(result)


schema = get_schema(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)
base_schema = format_schema(schema)

# 실제 데이터 예시를 포함한 enriched schema
ENRICHED_SCHEMA = base_schema + """

=== 실제 저장된 값 예시 (Cypher 생성 시 반드시 참고) ===

[Hall 노드 주요 속성]
- name: 홀 이름 전체 (예: '아모리스 역삼점', '더채플앳청담', '엘타워', 'JW Marriott 서울_서초')
  * 이름에 지역명이 포함된 경우 많음 (역삼, 청담, 강남, 서초 등)
- region: 광역 지역 (예: '서울', '경기', '부산', '서울 강남구', '경기 성남시')
  * 주의: '서울특별시' 가 아닌 '서울' 로 저장됨
- subRegion: 구/시 단위 (예: '강남구', '서초구', '마포구', '파주시')
- address: 주소 (예: '서울 강남구', '서울 서초구', '경기 성남시')
  * 상세 도로명 없이 시/구 수준으로만 저장된 경우 많음
- rating: 평점 0~100 (예: 82.0, 95.0)
- memoContent: 홀 상세 설명 텍스트
  * 예: "스타일: 어두운 / 컨벤션", "역삼역 도보 3분", "야외 정원 보유"
  * 특징/분위기 검색 시 CONTAINS 사용
- minIndividualHallPrice / maxIndividualHallPrice: 홀 전체 대관료 (원 단위)
  * '대관료', '홀대관료', '대관 비용' 등 사용자가 대관료를 물으면 반드시 minIndividualHallPrice 사용
- minRentalPrice / maxRentalPrice: 예식 렌탈료 (홀 대관료와 별개 항목)
- minMealPrice / maxMealPrice: 식대 (원 단위)
- memoContent: 최대 수용인원, 보증인원, 스타일, 예식형태 등 텍스트로 포함
  * 예: '최대 수용인원: 400명', '보증인원: 300명', '역삼역 도보 3분'

[Region 노드]
- name 예시: '서울', '경기', '부산', '대구', '서울 강남구', '서울 서초구', '경기 성남시'
  * 광역시/도 단위이거나 '서울 강남구' 처럼 구 단위까지 포함된 경우도 있음

[District 노드]
- name 예시: '강남구', '서초구', '마포구', '종로구', '파주시', '고양시'
  * 구(區) 또는 시(市) 단위

[StyleFilter 노드]
- name 전체 목록: '우아한', '심플한', '러블리', '깨끗/화사', '음영', '내추럴', '인물중심', '인물+배경', '빈티지', '클래식', '할인'

=== 위치 검색 규칙 ===
1. "역삼", "청담", "홍대", "합정" 같은 동/역 이름
   → Region이나 District에 없음 → h.name CONTAINS '역삼' OR h.address CONTAINS '역삼' 사용
2. "강남구", "서초구" 같은 구 단위
   → MATCH (h:Hall)-[:IN_DISTRICT]->(d:District) WHERE d.name CONTAINS '강남'
   → 또는 h.subRegion CONTAINS '강남'
3. "서울", "부산" 같은 광역 단위
   → h.region CONTAINS '서울' 또는 MATCH (h:Hall)-[:IN_REGION]->(r:Region) WHERE r.name CONTAINS '서울'
4. 특성/분위기 검색 (야외, 정원, 뷔페, 코스, 모던 등)
   → h.memoContent CONTAINS '야외'
"""

neo4j_schema = ENRICHED_SCHEMA

print("=== Neo4j Schema (Enriched) ===")
print(neo4j_schema[:500])
print("...")


=== Neo4j Schema ===
Node properties:
Image {url: STRING, title: STRING}
Hall {partnerId: INTEGER, name: STRING, address: STRING, profileUrl: STRING, rating: FLOAT, region: STRING, reviewCnt: INTEGER, tel: STRING, typeName: STRING, uuid: STRING, minRentalPrice: INTEGER, subRegion: STRING, partnerProfileId: INTEGER, modTsp: STRING, minMealPrice: INTEGER, partnerProfileName: STRING, minIndividualHallPrice: INTEGER, maxRentalPrice: INTEGER, bookingState: INTEGER, maxMealPrice: INTEGER, memoContent: STRING, partnerProfileUuid: STRING, availableContract: INTEGER, maxIndividualHallPrice: INTEGER}
EventTag {code: STRING}
District {name: STRING}
Tag {name: STRING, typeName: STRING, type: INTEGER, category: STRING}
Region {name: STRING}
Vendor {partnerId: INTEGER, name: STRING, category: STRING, addcostStr: STRING, address: STRING, coverUrl: STRING, detailCmt: STRING, enterpriseCode: STRING, eventOptionPrice: INTEGER, eventPrice: INTEGER, holiday: STRING, iweddingNo: STRING, likeCnt: INTEGER, o

## 선호도 기반 추천 테스트

In [74]:
cypher_query = """
MATCH (h:Hall)
OPTIONAL MATCH (h)-[:HAS_STYLE_FILTER]->(sf:StyleFilter)
OPTIONAL MATCH (h)-[:HAS_TAG]->(t:Tag)
OPTIONAL MATCH (h)-[:IN_REGION]->(r:Region)

WITH h,
     collect(DISTINCT sf.name) AS styles,
     collect(DISTINCT t.name) AS tags,
     collect(DISTINCT r.name) AS regions

WITH h, styles, tags, regions,
     CASE WHEN any(x IN styles WHERE x CONTAINS $style) THEN 2 ELSE 0 END +
     CASE WHEN any(x IN tags WHERE x CONTAINS $mood) OR h.memoContent CONTAINS $mood THEN 1 ELSE 0 END +
     CASE WHEN any(x IN regions WHERE x CONTAINS $region) THEN 2 ELSE 0 END +
     CASE WHEN any(x IN tags WHERE x CONTAINS $food) THEN 1 ELSE 0 END
     AS score

WHERE score > 0

RETURN h.partnerId AS hallId,
       h.name AS name,
       h.rating AS rating,
       h.reviewCnt AS reviewCnt,
       score
ORDER BY score DESC, rating DESC, reviewCnt DESC
LIMIT 10
"""

with driver.session() as session:
    result = session.run(
        cypher_query,
        style=str(pref.get("style", "")),
        mood=str(pref.get("mood", "")),
        food=str(pref.get("food", "")),
        region=str(pref.get("venue", pref.get("region", "")))
    )
    halls = [record.data() for record in result]

print("=== 선호 기반 추천 결과 ===")
print(halls)

=== 선호 기반 추천 결과 ===
[{'hallId': 1182, 'name': 'KDW웨딩', 'rating': 90.0, 'reviewCnt': 606, 'score': 6}, {'hallId': 12493, 'name': 'HW컨벤션센터', 'rating': 90.0, 'reviewCnt': 17, 'score': 6}, {'hallId': 4212, 'name': '서울웨딩타워', 'rating': 88.0, 'reviewCnt': 621, 'score': 6}, {'hallId': 11958, 'name': '메리스에이프럴', 'rating': 88.0, 'reviewCnt': 244, 'score': 6}, {'hallId': 1283, 'name': '엘컨벤션', 'rating': 87.0, 'reviewCnt': 59, 'score': 6}, {'hallId': 2482, 'name': '포레스트원 웨딩', 'rating': 86.0, 'reviewCnt': 115, 'score': 6}, {'hallId': 11380, 'name': '그래머시 코엑스_강남', 'rating': 86.0, 'reviewCnt': 15, 'score': 6}, {'hallId': 12113, 'name': '웨스턴베니비스 영등포점_영등포', 'rating': 85.0, 'reviewCnt': 51, 'score': 6}, {'hallId': 9652, 'name': 'BNT CONVENTION 웨딩', 'rating': 85.0, 'reviewCnt': 42, 'score': 6}, {'hallId': 3573, 'name': 'JW Marriott 동대문 스퀘어', 'rating': 85.0, 'reviewCnt': 17, 'score': 6}]


### 예시

In [75]:
examples = [
    """USER INPUT: '강남구에 있는 결혼식장 추천해줄래'
QUERY:
MATCH (h:Hall)-[:IN_DISTRICT]->(d:District)
WHERE d.name CONTAINS '강남'
RETURN h.partnerId AS hallId, h.name AS name, h.rating AS rating, h.reviewCnt AS reviewCnt, d.name AS district
ORDER BY h.rating DESC, h.reviewCnt DESC
LIMIT 10
""",

    """USER INPUT: '서울 강남 웨딩홀 추천해줘'
QUERY:
MATCH (h:Hall)-[:IN_DISTRICT]->(d:District)
WHERE d.name CONTAINS '강남'
RETURN h.partnerId AS hallId, h.name AS name, h.rating AS rating, h.reviewCnt AS reviewCnt, d.name AS district
ORDER BY h.reviewCnt DESC, h.rating DESC
LIMIT 10
""",

    """USER INPUT: '서울 종로구 결혼식장 추천'
QUERY:
MATCH (h:Hall)-[:IN_DISTRICT]->(d:District)
WHERE d.name CONTAINS '종로'
RETURN h.partnerId AS hallId, h.name AS name, h.rating AS rating, h.reviewCnt AS reviewCnt, d.name AS district
ORDER BY h.rating DESC, h.reviewCnt DESC
LIMIT 10
""",

    """USER INPUT: '호텔 웨딩홀 추천해줘'
QUERY:
MATCH (h:Hall)
OPTIONAL MATCH (h)-[:HAS_TAG]->(t:Tag)
OPTIONAL MATCH (h)-[:HAS_STYLE_FILTER]->(sf:StyleFilter)
WHERE t.name CONTAINS '호텔' OR sf.name CONTAINS '호텔' OR h.typeName CONTAINS '호텔'
RETURN h.partnerId AS hallId, h.name AS name, h.rating AS rating, h.reviewCnt AS reviewCnt
ORDER BY h.rating DESC, h.reviewCnt DESC
LIMIT 10
""",

    """USER INPUT: '웨딩홀 추천해줘'
QUERY:
MATCH (h:Hall)
WHERE h.rating IS NOT NULL
RETURN h.partnerId AS hallId, h.name AS name, h.rating AS rating, h.reviewCnt AS reviewCnt, h.region AS region
ORDER BY h.rating DESC, h.reviewCnt DESC
LIMIT 10
""",

    """USER INPUT: '럭셔리한 웨딩홀 추천해줘'
QUERY:
MATCH (h:Hall)
OPTIONAL MATCH (h)-[:HAS_TAG]->(t:Tag)
OPTIONAL MATCH (h)-[:HAS_STYLE_FILTER]->(sf:StyleFilter)
WHERE t.name CONTAINS '럭셔리' OR sf.name CONTAINS '럭셔리'
   OR t.name CONTAINS '화이트' OR sf.name CONTAINS '화이트'
RETURN h.partnerId AS hallId, h.name AS name, h.rating AS rating, h.reviewCnt AS reviewCnt
ORDER BY h.rating DESC, h.reviewCnt DESC
LIMIT 10
""",

    """USER INPUT: '야외 정원이 있는 웨딩홀 알려줘'
QUERY:
MATCH (h:Hall)
OPTIONAL MATCH (h)-[:HAS_TAG]->(t:Tag)
WHERE t.name CONTAINS '야외' OR t.name CONTAINS '정원' OR h.memoContent CONTAINS '야외'
RETURN h.partnerId AS hallId, h.name AS name, h.rating AS rating, h.reviewCnt AS reviewCnt
ORDER BY h.rating DESC, h.reviewCnt DESC
LIMIT 10
""",

    """USER INPUT: '평점 높은 웨딩홀 추천해줘'
QUERY:
MATCH (h:Hall)
WHERE h.rating IS NOT NULL AND h.rating > 0
RETURN h.partnerId AS hallId, h.name AS name, h.rating AS rating, h.reviewCnt AS reviewCnt, h.region AS region
ORDER BY h.rating DESC
LIMIT 10
""",

    """USER INPUT: '노보텔 앰버서더와 비슷한 웨딩홀 추천해줘'
QUERY:
MATCH (h:Hall)
OPTIONAL MATCH (h)-[:HAS_TAG]->(t:Tag)
WHERE t.name CONTAINS '호텔' OR t.name CONTAINS '럭셔리' OR h.typeName CONTAINS '호텔'
RETURN h.partnerId AS hallId, h.name AS name, h.rating AS rating, h.reviewCnt AS reviewCnt, h.region AS region
ORDER BY h.rating DESC, h.reviewCnt DESC
LIMIT 10
""",

    """USER INPUT: '부산 웨딩홀 추천해줘'
QUERY:
MATCH (h:Hall)
WHERE h.region CONTAINS '부산'
RETURN h.partnerId AS hallId, h.name AS name, h.rating AS rating, h.reviewCnt AS reviewCnt, h.region AS region
ORDER BY h.rating DESC, h.reviewCnt DESC
LIMIT 10
""",

    """USER INPUT: '리뷰 많은 웨딩홀 5개 추천해줘'
QUERY:
MATCH (h:Hall)
WHERE h.reviewCnt IS NOT NULL AND h.reviewCnt > 0
RETURN h.partnerId AS hallId, h.name AS name, h.rating AS rating, h.reviewCnt AS reviewCnt, h.region AS region
ORDER BY h.reviewCnt DESC
LIMIT 5
""",

    """USER INPUT: '역삼 근처 웨딩홀 추천해줘'
QUERY:
MATCH (h:Hall)
WHERE h.name CONTAINS '역삼'
   OR h.subRegion CONTAINS '역삼'
   OR h.address CONTAINS '역삼'
RETURN h.partnerId AS hallId, h.name AS name, h.rating AS rating, h.reviewCnt AS reviewCnt, h.region AS region, h.address AS address
ORDER BY h.rating DESC, h.reviewCnt DESC
LIMIT 10
""",

    """USER INPUT: '홍대 웨딩홀 추천해줘'
QUERY:
MATCH (h:Hall)
WHERE h.name CONTAINS '홍대'
   OR h.subRegion CONTAINS '홍대'
   OR h.subRegion CONTAINS '마포'
   OR h.address CONTAINS '홍대'
RETURN h.partnerId AS hallId, h.name AS name, h.rating AS rating, h.reviewCnt AS reviewCnt, h.region AS region
ORDER BY h.rating DESC, h.reviewCnt DESC
LIMIT 10
""",

    """USER INPUT: '강남역 웨딩홀 추천해줘'
QUERY:
MATCH (h:Hall)
OPTIONAL MATCH (h)-[:IN_DISTRICT]->(d:District)
WHERE h.name CONTAINS '강남'
   OR h.address CONTAINS '강남'
   OR (d IS NOT NULL AND d.name CONTAINS '강남')
RETURN h.partnerId AS hallId, h.name AS name, h.rating AS rating, h.reviewCnt AS reviewCnt, h.address AS address
ORDER BY h.rating DESC, h.reviewCnt DESC
LIMIT 10
""",


    """USER INPUT: '강남에 자연채광 있는 모던한 웨딩홀 추천해줘'
QUERY:
MATCH (h:Hall)
OPTIONAL MATCH (h)-[:IN_DISTRICT]->(d:District)
WHERE (d IS NULL OR d.name CONTAINS '강남')
  AND (h.memoContent CONTAINS '자연채광' OR h.memoContent CONTAINS '모던' OR h.memoContent CONTAINS '밝은')
RETURN h.partnerId AS hallId, h.name AS name, h.rating AS rating, h.reviewCnt AS reviewCnt, h.subRegion AS subRegion, h.memoContent AS memo
ORDER BY h.rating DESC
LIMIT 10
""",

    """USER INPUT: '인터컨티넨탈 웨딩홀 알려줘'
QUERY:
MATCH (h:Hall)
WHERE h.name CONTAINS '컨티넨탈' OR h.name CONTAINS '컨티넨털' OR h.name CONTAINS 'Continental' OR h.name CONTAINS '인터컨'
RETURN h.partnerId AS hallId, h.name AS name, h.rating AS rating, h.reviewCnt AS reviewCnt, h.address AS address
LIMIT 10
""",

    """USER INPUT: '야외 정원 웨딩 홀 추천해줘'
QUERY:
MATCH (h:Hall)
WHERE h.memoContent CONTAINS '야외' OR h.memoContent CONTAINS '정원' OR h.memoContent CONTAINS '가든'
RETURN h.partnerId AS hallId, h.name AS name, h.rating AS rating, h.reviewCnt AS reviewCnt, h.region AS region, h.memoContent AS memo
ORDER BY h.rating DESC
LIMIT 10
""",,

    """USER INPUT: '엘타워 대관료 알려줘'
QUERY:
MATCH (h:Hall)
WHERE h.name CONTAINS '엘타워'
RETURN h.name AS name, h.minIndividualHallPrice AS minHallPrice,
       h.maxIndividualHallPrice AS maxHallPrice,
       h.minMealPrice AS minMealPrice, h.maxMealPrice AS maxMealPrice
LIMIT 1
""",

    """USER INPUT: '아펠가모 반포점 수용인원 알려줘'
QUERY:
MATCH (h:Hall)
WHERE h.name CONTAINS '아펠가모 반포'
RETURN h.name AS name, h.memoContent AS memoContent
LIMIT 1
""",
]


In [76]:
llm = OpenAILLM(
    model_name="gpt-4o",
    model_params={"temperature": 0}
)

In [77]:
def recommend_by_preference(driver, pref):
    region = str(pref.get("venue") or pref.get("region") or "")
    style = str(pref.get("style") or "")
    mood_raw = str(pref.get("mood") or "")
    food = str(pref.get("food") or "")
    budget_raw = str(pref.get("budget") or "")
    guest_raw = str(pref.get("guest_count") or "")

    # 분위기 여러 개 분리 (콤마 구분)
    moods = [m.strip() for m in mood_raw.split(",") if m.strip()]

    # 예산 파싱 (예: "300만원" → 3000000)
    import re
    budget_nums = [int(x) for x in re.findall(r"\d+", budget_raw)]
    max_meal_budget = budget_nums[0] * 10000 if budget_nums else None

    # 게스트 수 파싱
    guest_nums = [int(x) for x in re.findall(r"\d+", guest_raw)]
    guest_count = guest_nums[0] if guest_nums else None

    cypher_query = """
    MATCH (h:Hall)

    WITH h,
         CASE WHEN $region <> '' AND (h.region CONTAINS $region OR h.address CONTAINS $region OR h.subRegion CONTAINS $region) THEN 3 ELSE 0 END AS region_score,
         CASE WHEN $style <> '' AND h.memoContent CONTAINS $style THEN 2 ELSE 0 END AS style_score,
         CASE WHEN $food <> '' AND (h.memoContent CONTAINS $food) THEN 1 ELSE 0 END AS food_score,
         CASE WHEN $guest_count IS NULL OR (h.memoContent IS NOT NULL) THEN 0 ELSE 0 END AS guest_score

    WITH h, (region_score + style_score + food_score) AS score

    WHERE ($region = '' OR h.region CONTAINS $region OR h.address CONTAINS $region)

    RETURN h.partnerId AS hallId,
           h.name AS name,
           h.rating AS rating,
           h.reviewCnt AS reviewCnt,
           h.region AS region,
           h.subRegion AS subRegion,
           h.minMealPrice AS minMealPrice,
           h.minIndividualHallPrice AS minHallPrice,
           score
    ORDER BY score DESC, h.rating DESC, h.reviewCnt DESC
    LIMIT 10
    """

    with driver.session() as session:
        result = session.run(
            cypher_query,
            region=region,
            style=style,
            food=food,
            guest_count=guest_count,
        )
        rows = [record.data() for record in result]

    # 분위기 키워드 보너스 점수 (Python에서 처리)
    if moods:
        for row in rows:
            memo = row.get("memoContent") or ""
            bonus = sum(1 for m in moods if m in memo)
            row["score"] = (row.get("score") or 0) + bonus
        rows.sort(key=lambda x: (-x.get("score", 0), -(x.get("rating") or 0), -(x.get("reviewCnt") or 0)))

    return rows


In [78]:
def recommend_by_facility(driver, keyword, region=None):

    cypher_query = """
    MATCH (h:Hall)-[:IN_REGION]->(r:Region)

    WHERE h.memoContent CONTAINS $keyword
      AND ($region IS NULL OR r.name CONTAINS $region)

    RETURN h.partnerId AS hallId,
           h.name AS name,
           r.name AS region,
           h.rating AS rating,
           h.reviewCnt AS reviewCnt

    ORDER BY rating DESC, reviewCnt DESC
    LIMIT 10
    """

    with driver.session() as session:
        result = session.run(
            cypher_query,
            keyword=keyword,
            region=region
        )

        return [record.data() for record in result]

In [79]:
def extract_region(message: str):
    msg = message.strip()

    region_patterns = [
        r"(서울|경기|인천|부산|대구|대전|광주|울산|세종|제주)\s*(지역|쪽)?",
    ]

    for pattern in region_patterns:
        m = re.search(pattern, msg)
        if m:
            return m.group(1)

    return None

In [80]:
def extract_facility_keyword(message: str) -> str:
    if "주차" in message or "주차장" in message:
        return "주차"
    if "발렛" in message or "발렛파킹" in message:
        return "발렛"
    if "역세권" in message:
        return "역"
    if "보증인원" in message:
        return "보증인원"
    if "코스" in message:
        return "코스"
    if "뷔페" in message:
        return "뷔페"
    return ""

In [81]:
def kakao_search_keyword(query: str):
    import os
    import requests

    api_key = os.environ.get("KAKAO_REST_API_KEY")
    if not api_key:
        raise ValueError("KAKAO_REST_API_KEY가 설정되지 않았습니다.")

    url = "https://dapi.kakao.com/v2/local/search/keyword.json"
    headers = {"Authorization": f"KakaoAK {api_key.strip()}"}
    params = {"query": query}

    try:
        resp = requests.get(url, headers=headers, params=params, timeout=10)
    except requests.RequestException as e:
        print(f"[KAKAO keyword 요청 실패] query={query}, error={e}")
        return None

    if resp.status_code == 429:
        print(f"[KAKAO keyword 429] query={query}")
        return None

    if resp.status_code != 200:
        print(f"[KAKAO keyword 오류] status={resp.status_code}, query={query}")
        print(resp.text)
        return None

    data = resp.json()
    docs = data.get("documents", [])
    if not docs:
        return None

    first = docs[0]
    return {
        "x": float(first["x"]),
        "y": float(first["y"]),
        "place_name": first.get("place_name"),
        "address_name": first.get("address_name"),
    }

In [82]:
def kakao_search_address(query: str):
    import os
    import requests

    api_key = os.environ.get("KAKAO_REST_API_KEY")
    if not api_key:
        raise ValueError("KAKAO_REST_API_KEY가 설정되지 않았습니다.")

    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {api_key.strip()}"}
    params = {"query": query}

    try:
        resp = requests.get(url, headers=headers, params=params, timeout=10)
    except requests.RequestException as e:
        print(f"[KAKAO address 요청 실패] query={query}, error={e}")
        return None

    if resp.status_code == 429:
        print(f"[KAKAO address 429] query={query}")
        return None

    if resp.status_code != 200:
        print(f"[KAKAO address 오류] status={resp.status_code}, query={query}")
        print(resp.text)
        return None

    data = resp.json()
    docs = data.get("documents", [])
    if not docs:
        return None

    first = docs[0]
    return {
        "x": float(first["x"]),
        "y": float(first["y"]),
        "address_name": first.get("address_name"),
    }

In [83]:
geo_cache = {}

def geocode_place(query: str):
    query = (query or "").strip()
    if not query:
        return None

    if query in geo_cache:
        return geo_cache[query]

    result = kakao_search_address(query)
    if result:
        coord = (result["y"], result["x"])
        geo_cache[query] = coord
        return coord

    result = kakao_search_keyword(query)
    if result:
        coord = (result["y"], result["x"])
        geo_cache[query] = coord
        return coord

    geo_cache[query] = None
    return None

In [84]:
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)

    a = sin(dlat / 2) ** 2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2) ** 2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

In [85]:
def search_halls_for_location(driver, region=None, districts=None, keyword=None, limit=500):
    query = """MATCH (h:Hall)
WHERE ($region IS NULL
       OR h.region = $region
       OR h.address STARTS WITH $region)
  AND (
        size($districts) = 0
        OR any(x IN $districts WHERE h.address CONTAINS x OR h.subRegion CONTAINS x)
  )
RETURN DISTINCT
       h.partnerId AS hallId,
       h.name AS name,
       h.address AS address,
       h.region AS region,
       h.subRegion AS district,
       h.rating AS rating,
       h.reviewCnt AS reviewCnt,
       h.minIndividualHallPrice AS minHallPrice,
       h.maxIndividualHallPrice AS maxHallPrice,
       h.minMealPrice AS minMealPrice,
       h.maxMealPrice AS maxMealPrice,
       CASE WHEN $keyword IS NOT NULL AND (h.name CONTAINS $keyword OR h.subRegion CONTAINS $keyword)
            THEN true ELSE false END AS name_match
ORDER BY name_match DESC
LIMIT $limit"""

    with driver.session() as session:
        result = session.run(
            query,
            region=region,
            districts=districts or [],
            keyword=keyword,
            limit=limit
        )
        rows = [record.data() for record in result]

    # 파라미터 치환한 표시용 쿼리 생성
    display_query = query
    display_query += f"""
// 파라미터: region={repr(region)}, districts={districts or []}, keyword={repr(keyword)}, limit={limit}"""

    return rows, display_query


In [86]:
def filter_halls_by_region_and_district(halls, region=None, districts=None):
    filtered = []

    for h in halls:
        address = (h.get("address") or "").strip()
        district = (h.get("district") or "").strip()
        hall_region = (h.get("region") or "").strip()

        if region:
            if not (address.startswith(region) or hall_region == region):
                continue

        if districts:
            if not any(d in address or d in district for d in districts):
                continue

        filtered.append(h)

    return filtered

In [87]:
def infer_region_from_start_location(start_location: str):
    if not start_location:
        return None, []

    if any(x in start_location for x in ["역삼", "강남", "선릉", "삼성", "청담", "압구정", "잠실"]):
        return "서울", []

    if any(x in start_location for x in ["판교", "분당", "수원", "용인"]):
        return "경기", []

    return None, []

In [88]:
def calculate_distance_and_sort(start_location: str, halls: list[dict]):
    start_coord = geocode_place(start_location)
    if not start_coord:
        return []

    start_lat, start_lng = start_coord

    # 1. 구별 주소 캐싱 - distinct address만 geocode (API 호출 최소화)
    address_cache = {}
    for hall in halls:
        addr = hall.get("address") or ""
        if addr and addr not in address_cache:
            coord = geocode_place(addr)
            address_cache[addr] = coord

    # 2. name_match 홀은 이름으로 정밀 geocode (소수만)
    name_coord_cache = {}
    for hall in halls:
        if hall.get("name_match"):
            name = hall.get("name", "")
            if name and name not in name_coord_cache:
                coord = geocode_place(name)
                name_coord_cache[name] = coord

    ranked = []
    for hall in halls:
        # name_match 홀: 이름 좌표 우선
        hall_coord = None
        if hall.get("name_match"):
            hall_coord = name_coord_cache.get(hall.get("name", ""))
        # 그 외: 캐싱된 주소 좌표 사용
        if not hall_coord:
            hall_coord = address_cache.get(hall.get("address") or "")
        if not hall_coord:
            continue

        hall_lat, hall_lng = hall_coord
        distance_km = haversine_distance(start_lat, start_lng, hall_lat, hall_lng)

        item = dict(hall)
        # name_match 홀이고 이름으로 정밀 좌표를 얻었으면 해당 거리 사용
        # name_match지만 이름 geocode 실패 시 주소 거리에 -0.1 보정
        if hall.get("name_match"):
            if name_coord_cache.get(hall.get("name", "")):
                item["distance_km"] = round(distance_km, 2)
            else:
                item["distance_km"] = round(max(0.01, distance_km - 0.5), 2)
        else:
            item["distance_km"] = round(distance_km, 2)

        ranked.append(item)

    ranked.sort(
        key=lambda x: (
            x["distance_km"],
            -(x.get("rating") or 0),
            -(x.get("reviewCnt") or 0)
        )
    )
    return ranked


In [89]:
def extract_start_location(message: str):
    msg = message.strip()

    patterns = [
        r"(.+?)에서\s*(가까운|근처|인근|몇\s*분|도보|차로|이내)",
        r"(.+?)기준으로",
        r"(.+?)\s*근처(?:로)?",
        r"(.+?)\s*인근",
        r"(.+?)\s*주변",
        r"(.+?)\s*가까운\s*곳",
    ]

    for pattern in patterns:
        m = re.search(pattern, msg)
        if m:
            location = m.group(1).strip()

            location = re.sub(
                r"(웨딩홀|예식장|추천|알려줘|찾아줘)$",
                "",
                location
            ).strip()

            if len(location) < 2:
                return None

            return location

    return None

In [90]:
def extract_location_with_llm(message: str) -> str | None:
    prompt = f"""
사용자 질문에서 출발 위치만 추출하세요.

규칙:
- 역, 지역, 동네, 랜드마크 등 위치만 추출
- 없으면 null 반환
- 설명하지 말고 JSON만 출력

예시

입력: 역삼역에서 가까운 웨딩홀
출력: {{"location": "역삼역"}}

입력: 강남 근처 웨딩홀
출력: {{"location": "강남"}}

입력: 서울에서 유명한 웨딩홀
출력: {{"location": "서울"}}

사용자 질문:
{message}
"""

    try:
        result = llm.invoke(prompt).content.strip()

        import json
        parsed = json.loads(result)

        location = parsed.get("location")

        if location and location.lower() != "null":
            return location.strip()

        return None

    except Exception as e:
        print("location extract error:", e)
        return None

In [91]:
def rerank_location_candidates_with_llm(message: str, halls: list[dict]) -> list[dict]:
    if not halls:
        return []

    hall_lines = []
    for idx, h in enumerate(halls, start=1):
        hall_lines.append(
            f"{idx}. 이름: {h.get('name', '')}, "
            f"주소: {h.get('address', '')}, "
            f"지역: {h.get('region', '')}, "
            f"구: {h.get('district', '')}, "
            f"평점: {h.get('rating', 0)}, "
            f"리뷰수: {h.get('reviewCnt', 0)}"
        )

    prompt = f"""
당신은 웨딩홀 추천 보조 시스템입니다.

사용자 질문:
{message}

후보 웨딩홀 목록:
{chr(10).join(hall_lines)}

작업:
1. 사용자 질문에 맞지 않는 후보는 제외하세요.
2. 위치 질문이면 같은 권역의 후보를 우선하세요.
3. 질문과 무관한 지역은 제외하세요.
4. 결과는 가장 적절한 순서대로 최대 5개만 고르세요.
5. JSON만 출력하세요.

형식:
{{
  "selected_indices": [1, 3, 5]
}}
"""
    try:
        result = llm.invoke(prompt).content.strip()
        parsed = json.loads(result)
        selected = parsed.get("selected_indices", [])

        reranked = []
        for i in selected:
            if 1 <= i <= len(halls):
                reranked.append(halls[i - 1])

        return reranked if reranked else halls[:5]

    except Exception as e:
        print("LLM rerank error:", e)
        return halls[:5]

In [92]:
def recommend_by_location(driver, conn, couple_id, message):
    start_location = extract_location_with_llm(message)
    region = extract_region(message)
    districts = []

    if not start_location or len(start_location) < 2:
        return {
            "error": "출발 위치를 이해하지 못했습니다. 예: 역삼역에서 가까운 웨딩홀 추천해줘"
        }

    if not region:
        inferred_region, inferred_districts = infer_region_from_start_location(start_location)
        region = inferred_region
        districts = inferred_districts

    halls, neo4j_query = search_halls_for_location(driver, region=region, districts=districts, keyword=start_location, limit=500)
    halls = filter_halls_by_region_and_district(halls, region=region, districts=districts)

    if not halls:
        return {
            "error": "조건에 맞는 웨딩홀을 찾지 못했습니다."
        }

    ranked = calculate_distance_and_sort(start_location, halls)

    if ranked:
        return {
            "mode": "distance",
            "start_location": start_location,
            "region": region,
            "results": ranked[:10],
            "neo4j_query": neo4j_query,
        }

    llm_ranked = rerank_location_candidates_with_llm(message, halls)

    return {
        "mode": "llm_fallback",
        "start_location": start_location,
        "region": region,
        "results": llm_ranked[:5],
        "neo4j_query": neo4j_query,
    }

In [93]:
print(geocode_place("역삼역"))
print(geocode_place("서울 강남구 테헤란로"))

(37.5006744185994, 127.03646946847)
(37.504059366187, 127.047486752713)


In [94]:
def compare_two_halls(driver, hall_a: str, hall_b: str):
    cypher = """
    MATCH (a:Hall {name:$hall_a})
    MATCH (b:Hall {name:$hall_b})
    RETURN
        a.name AS hallAName,
        a.rating AS hallARating,
        a.reviewCnt AS hallAReviewCnt,
        a.minMealPrice AS hallAMinMealPrice,
        a.maxMealPrice AS hallAMaxMealPrice,
        a.minRentalPrice AS hallAMinRentalPrice,
        a.maxRentalPrice AS hallAMaxRentalPrice,
        b.name AS hallBName,
        b.rating AS hallBRating,
        b.reviewCnt AS hallBReviewCnt,
        b.minMealPrice AS hallBMinMealPrice,
        b.maxMealPrice AS hallBMaxMealPrice,
        b.minRentalPrice AS hallBMinRentalPrice,
        b.maxRentalPrice AS hallBMaxRentalPrice
    """
    with driver.session() as session:
        result = session.run(cypher, hall_a=hall_a, hall_b=hall_b)
        return [record.data() for record in result]

In [95]:
def extract_multiple_halls(message: str):
    """LLM을 사용해 메시지에서 웨딩홀 이름 추출 (regex 오파싱 방지)"""
    prompt = f"""다음 메시지에서 비교 대상인 웨딩홀/예식장 이름만 추출하세요.
JSON 배열로만 반환하세요 (설명 없이, 마크다운 없이).

메시지: {message}

예시:
입력: "아모리스 역삼점이랑 JW Marriott 서울 이거 두개 비교"
출력: ["아모리스 역삼점", "JW Marriott 서울"]

입력: "노보텔 앰버서더와 삼정호텔 비교해줘"
출력: ["노보텔 앰버서더", "삼정호텔"]

입력: "엘타워, 더채플앳청담, 아펠가모 반포점 비교"
출력: ["엘타워", "더채플앳청담", "아펠가모 반포점"]
"""
    try:
        result = llm.invoke(prompt).content.strip()
        import re as _re
        match = _re.search(r'\[.*?\]', result, _re.DOTALL)
        if match:
            return json.loads(match.group())
    except Exception as e:
        print(f"extract_multiple_halls LLM error: {e}")
    return []


In [96]:
print(extract_multiple_halls("노보텔과 삼정호텔 비교해줘"))
# ['노보텔', '삼정호텔']

print(extract_multiple_halls("노보텔 앰버서더와 삼정호텔 비교해줘"))
# ['노보텔 앰버서더', '삼정호텔']

print(extract_multiple_halls("노보텔, 삼정호텔, 아펠가모 반포점 비교해줘"))
# ['노보텔', '삼정호텔', '아펠가모 반포점']

['노보텔', '삼정호텔']
['노보텔 앰버서더', '삼정호텔']
['노보텔', '삼정호텔', '아펠가모 반포점']


In [97]:
def resolve_hall_name(driver, keyword: str):
    cypher = """
    MATCH (h:Hall)
    WHERE h.name CONTAINS $keyword
       OR replace(h.name, " ", "") CONTAINS replace($keyword, " ", "")
    RETURN h.name AS name,
           h.rating AS rating,
           h.reviewCnt AS reviewCnt
    ORDER BY h.reviewCnt DESC, h.rating DESC
    LIMIT 1
    """
    with driver.session() as session:
        result = session.run(cypher, keyword=keyword)
        row = result.single()
        return row["name"] if row else None

In [98]:
def resolve_multiple_hall_names(driver, hall_keywords):
    resolved = []

    for keyword in hall_keywords:
        real_name = resolve_hall_name(driver, keyword)
        if real_name:
            resolved.append(real_name)

    # 중복 제거
    resolved = list(dict.fromkeys(resolved))
    return resolved

In [99]:
def compare_multiple_halls(driver, hall_names):
    cypher = """
    MATCH (h:Hall)
    WHERE h.name IN $hall_names
    RETURN
        h.name AS name,
        h.region AS region,
        h.subRegion AS subRegion,
        h.address AS address,
        h.rating AS rating,
        h.reviewCnt AS reviewCnt,
        h.minMealPrice AS minMealPrice,
        h.maxMealPrice AS maxMealPrice,
        h.minRentalPrice AS minRentalPrice,
        h.maxRentalPrice AS maxRentalPrice,
        h.minIndividualHallPrice AS minHallPrice,
        h.maxIndividualHallPrice AS maxHallPrice,
        h.memoContent AS memoContent,
        h.tel AS tel
    ORDER BY h.reviewCnt DESC, h.rating DESC
    """
    with driver.session() as session:
        result = session.run(cypher, hall_names=hall_names)
        return [record.data() for record in result]


In [100]:
def compare_multiple_halls(driver, hall_names):
    cypher = """
    MATCH (h:Hall)
    WHERE h.name IN $hall_names
    RETURN
        h.name AS name,
        h.region AS region,
        h.subRegion AS subRegion,
        h.rating AS rating,
        h.reviewCnt AS reviewCnt,
        h.minMealPrice AS minMealPrice,
        h.maxMealPrice AS maxMealPrice,
        h.minRentalPrice AS minRentalPrice,
        h.maxRentalPrice AS maxRentalPrice
    """

    with driver.session() as session:
        result = session.run(cypher, hall_names=hall_names)
        return [record.data() for record in result]

In [101]:
def make_multi_compare_answer(rows):
    if not rows:
        return "비교할 웨딩홀 정보를 찾지 못했습니다."

    lines = ["웨딩홀 비교 결과입니다.\n"]

    for idx, row in enumerate(rows, start=1):
        hall_price_min = row.get("minHallPrice")
        hall_price_max = row.get("maxHallPrice")
        rental_min = row.get("minRentalPrice")
        rental_max = row.get("maxRentalPrice")
        meal_min = row.get("minMealPrice")
        meal_max = row.get("maxMealPrice")

        def fmt_price(v):
            return f"{int(v):,}원" if v is not None else "정보없음"

        price_lines = []
        if hall_price_min is not None:
            price_lines.append(f"  · 홀 전체 대관료: {fmt_price(hall_price_min)} ~ {fmt_price(hall_price_max)}")
        if rental_min is not None:
            price_lines.append(f"  · 예식 렌탈료: {fmt_price(rental_min)} ~ {fmt_price(rental_max)}")
        if meal_min is not None:
            price_lines.append(f"  · 식대: {fmt_price(meal_min)} ~ {fmt_price(meal_max)}")

        price_str = "\n".join(price_lines) if price_lines else "  · 가격 정보없음"

        lines.append(
            f"{idx}. {row['name']}\n"
            f"- 지역: {row.get('region', '정보없음')} {row.get('subRegion', '')}\n"
            f"- 평점: {row.get('rating', '정보없음')} / 리뷰수: {row.get('reviewCnt', '정보없음')}\n"
            f"- 가격:\n{price_str}\n"
        )

    return "\n".join(lines)


## LLM / Retriever / GraphRAG

In [102]:
from neo4j_graphrag.generation import RagTemplate
from neo4j_graphrag.retrievers.text2cypher import Text2CypherTemplate

SYSTEM_INSTRUCTIONS = """당신은 웨딩홀 추천 챗봇입니다.
반드시 아래 규칙을 따르세요:
1. Context(데이터베이스 검색 결과)에 있는 웨딩홀을 우선적으로 추천하세요.
2. 사용자 조건을 모두 만족하는 홀이 없더라도, Context에 있는 가장 근접한 결과를 추천하세요.
   예: 강남에 자연채광 홀이 없으면 → 강남 홀 중 가장 평점 높은 것을 추천하면서 "자연채광 조건은 정확히 맞지 않지만..."이라고 안내
3. Context가 완전히 비어있더라도 "없습니다"로 끝내지 마세요. 조건을 완화한 대안을 제안하세요.
4. Context에 없는 웨딩홀을 새로 만들어내거나 추가하지 마세요.
5. 응답 마지막에 항상 추가 조건 조정 여부를 물어보세요.
"""

CYPHER_TEMPLATE = """Task: Generate a Cypher statement for querying a Neo4j graph database from a user input.

위치/특성 검색 규칙 (반드시 따를 것):
- 구 단위 지역(강남구, 서초구 등): MATCH (h:Hall)-[:IN_DISTRICT]->(d:District) WHERE d.name CONTAINS '강남'
- 역/동네 이름(역삼, 홍대 등): h.name CONTAINS '역삼' OR h.address CONTAINS '역삼'
- 광역 지역(서울, 부산 등): h.region CONTAINS '서울'
- 특성/분위기 검색(자연채광, 모던, 야외, 정원 등): h.memoContent CONTAINS '자연채광'
- 조건이 여러 개면 모든 조건 충족 못할 수 있으므로 OR로 넓게 검색하거나 OPTIONAL MATCH 활용
- 홀 이름 검색 시 유사 표기도 함께: h.name CONTAINS '컨티넨탈' OR h.name CONTAINS '컨티넨털' OR h.name CONTAINS 'Continental'

Hall 노드 주요 속성:
- name: 홀 이름
- rating: 평점 (0-100)
- reviewCnt: 리뷰 수
- region: 광역시/도 (예: '서울특별시')
- subRegion: 구 (예: '강남구')
- address: 주소
- memoContent: 홀 상세 설명 (스타일, 예식형태, 식사, 특징 등)
- typeName: 홀 타입

Schema:
{schema}

Examples (optional):
{examples}

Input:
{query_text}

Do not use any properties or relationships not included in the schema.
Do not include triple backticks ``` or any additional text except the generated Cypher statement in your response.

Cypher query:"""

rag_template = RagTemplate(system_instructions=SYSTEM_INSTRUCTIONS)

retriever = Text2CypherRetriever(
    driver=driver,
    llm=llm,
    neo4j_schema=neo4j_schema,
    examples=examples,
    custom_prompt=CYPHER_TEMPLATE,
)

rag = GraphRAG(
    retriever=retriever,
    llm=llm,
    prompt_template=rag_template,
)


### Retriever / RAG 테스트

In [103]:
query_text = "서울 강남의 괜찮은 웨딩홀 추천해줘"
search_result = retriever.search(query_text=query_text)

print("=== 생성된 Cypher ===")
print(search_result.metadata.get("cypher", ""))

print("=== 조회 결과 ===")
print(search_result.items)

query_text = "더리버사이드호텔 웨딩과 비슷한 웨딩홀 중 평점이 높은 웨딩홀은 어디인가요?"
response = rag.search(query_text=query_text, return_context=True)

print("=== RAG 생성 Cypher ===")
print(response.retriever_result.metadata.get("cypher", ""))

print("=== 최종 답변 ===")
print(response.answer)

=== 생성된 Cypher ===
MATCH (h:Hall)-[:IN_REGION]->(:Region {name:'서울'})
OPTIONAL MATCH (h)-[:IN_DISTRICT]->(d:District)
WHERE d.name CONTAINS '강남'
RETURN h.partnerId AS hallId, h.name AS name, h.rating AS rating, h.reviewCnt AS reviewCnt
ORDER BY rating DESC, reviewCnt DESC
LIMIT 10
=== 조회 결과 ===
[RetrieverResultItem(content="<Record hallId=1182 name='KDW웨딩' rating=90.0 reviewCnt=606>", metadata=None), RetrieverResultItem(content="<Record hallId=12493 name='HW컨벤션센터' rating=90.0 reviewCnt=17>", metadata=None), RetrieverResultItem(content="<Record hallId=4212 name='서울웨딩타워' rating=88.0 reviewCnt=621>", metadata=None), RetrieverResultItem(content="<Record hallId=11958 name='메리스에이프럴' rating=88.0 reviewCnt=244>", metadata=None), RetrieverResultItem(content="<Record hallId=1283 name='엘컨벤션' rating=87.0 reviewCnt=59>", metadata=None), RetrieverResultItem(content="<Record hallId=2482 name='포레스트원 웨딩' rating=86.0 reviewCnt=115>", metadata=None), RetrieverResultItem(content="<Record hallId=11380 name

### Gradio

In [104]:
def safe_str(x, max_len=6000):
    try:
        s = json.dumps(x, ensure_ascii=False, indent=2)
    except Exception:
        s = str(x)
    return s if len(s) <= max_len else s[:max_len] + "\n... (truncated)"


class Seafoam(gr.themes.Base):
    pass


seafoam = Seafoam()

In [105]:
raw_halls, _ = search_halls_for_location(driver, region="서울", districts=[], limit=10)
print(type(raw_halls))
print(len(raw_halls))
for h in raw_halls[:5]:
    print(h)

DEBUG NEW query: 
    MATCH (h:Hall)
    WHERE ($region IS NULL
           OR h.region = $region
           OR h.address STARTS WITH $region)
      AND (
            size($districts) = 0
            OR any(x IN $districts WHERE h.address CONTAINS x OR h.subRegion CONTAINS x)
      )
    RETURN DISTINCT
           h.partnerId AS hallId,
           h.name AS name,
           h.address AS address,
           h.region AS region,
           h.subRegion AS district,
           h.rating AS rating,
           h.reviewCnt AS reviewCnt
    LIMIT $limit
    
DEBUG NEW region param: '서울'
DEBUG NEW districts param: []
DEBUG NEW rows count: 10
<class 'list'>
10
{'hallId': 14117, 'name': '풀만 앰배서더 이스트폴 웨딩', 'address': '서울 광진구', 'region': '서울', 'district': '광진구', 'rating': 0.0, 'reviewCnt': None}
{'hallId': 1438, 'name': '그랜드 머큐어 임피리얼 팰리스 서울_강남', 'address': '서울 강남구', 'region': '서울', 'district': '강남구', 'rating': 76.0, 'reviewCnt': 22}
{'hallId': 1437, 'name': '더화이트베일_서초', 'address': '서울 서초구', 'region': 

In [106]:
result = recommend_by_location(driver, conn, COUPLE_ID, "역삼역에서 가까운 웨딩홀 추천해줘")
print(result)

DEBUG NEW query: 
    MATCH (h:Hall)
    WHERE ($region IS NULL
           OR h.region = $region
           OR h.address STARTS WITH $region)
      AND (
            size($districts) = 0
            OR any(x IN $districts WHERE h.address CONTAINS x OR h.subRegion CONTAINS x)
      )
    RETURN DISTINCT
           h.partnerId AS hallId,
           h.name AS name,
           h.address AS address,
           h.region AS region,
           h.subRegion AS district,
           h.rating AS rating,
           h.reviewCnt AS reviewCnt
    LIMIT $limit
    
DEBUG NEW region param: '서울'
DEBUG NEW districts param: []
DEBUG NEW rows count: 100
{'mode': 'distance', 'start_location': '역삼역', 'region': '서울', 'results': [{'hallId': 760, 'name': '더리버사이드호텔 웨딩', 'address': '서울 서초구', 'region': '서울', 'district': '서초구', 'rating': 80.0, 'reviewCnt': 688, 'distance_km': 1.93}, {'hallId': 1237, 'name': '아펠가모 반포점', 'address': '서울 서초구', 'region': '서울', 'district': '서초구', 'rating': 79.0, 'reviewCnt': 245, 'distance

### Gradio App

In [ ]:
with gr.Blocks(theme=seafoam) as demo:

    def save_message_to_db(couple_id, role, content, intent=None):
        """대화 메시지를 MySQL에 저장"""
        try:
            c = conn.cursor()
            c.execute(
                "INSERT INTO chat_history (couple_id, role, content, intent) VALUES (%s, %s, %s, %s)",
                (couple_id, role, content[:4000], intent)
            )
            conn.commit()
            c.close()
        except Exception as e:
            print(f"DB 저장 오류: {e}")

    def load_history_from_db(couple_id, limit=20):
        """MySQL에서 최근 대화 기록 불러오기"""
        try:
            c = conn.cursor()
            c.execute(
                """SELECT role, content FROM chat_history
                   WHERE couple_id = %s
                   ORDER BY created_at DESC LIMIT %s""",
                (couple_id, limit)
            )
            rows = c.fetchall()
            c.close()
            # 최신순으로 가져왔으니 역순으로
            return [{"role": r, "content": c} for r, c in reversed(rows)]
        except Exception as e:
            print(f"DB 불러오기 오류: {e}")
            return []

    def build_context_from_db_history(couple_id, max_turns=6):
        """DB 기록을 LLM 프롬프트용 문자열로 변환"""
        history = load_history_from_db(couple_id, limit=max_turns * 2)
        if not history:
            return ""
        lines = []
        for m in history:
            role = "사용자" if m["role"] == "user" else "어시스턴트"
            lines.append(f"{role}: {m['content']}")
        return "\n".join(lines)

    def format_history_for_prompt(chat_history, max_turns=5):
        """최근 N턴의 대화 기록을 프롬프트용 문자열로 변환"""
        if not chat_history:
            return ""
        recent = chat_history[-(max_turns * 2):]
        lines = []
        for m in recent:
            role = "사용자" if m["role"] == "user" else "어시스턴트"
            lines.append(f"{role}: {m['content']}")
        return "\n".join(lines)

    def extract_hall_names_from_history(chat_history):
        """이전 어시스턴트 답변에서 언급된 웨딩홀 이름 목록 추출 (LLM 사용)"""
        if not chat_history:
            return []
        last_assistant = ""
        for m in reversed(chat_history):
            if m["role"] == "assistant":
                last_assistant = m["content"]
                break
        if not last_assistant:
            return []

        prompt = f"""아래 텍스트에서 언급된 웨딩홀 또는 예식장 이름만 JSON 배열로 추출하세요.
규칙:
- 홀/예식장/호텔 이름만 추출
- 없으면 빈 배열 반환
- 오직 JSON만 출력 (설명 없이)

텍스트:
{last_assistant}

출력 예시: ["그랜드 하얏트 서울", "롯데호텔 서울"]
"""
        try:
            result = llm.invoke(prompt).content.strip()
            import re as _re
            match = _re.search(r'\[.*?\]', result, _re.DOTALL)
            if match:
                return json.loads(match.group())
        except Exception:
            pass
        return []

    def parse_intent(message: str, chat_history=None) -> dict:
        """LLM으로 의도와 조건을 한 번에 파싱"""
        history_text = format_history_for_prompt(chat_history, max_turns=3) if chat_history else ""
        history_section = f"[이전 대화]\n{history_text}\n\n" if history_text else ""

        prompt = f"""{history_section}[현재 사용자 메시지]
{message}

위 대화를 분석해서 아래 JSON 형식으로만 응답하세요. 설명 없이 JSON만 출력하세요.

{{
  "intent": "LOCATION | NEO4J_RAG | COMPARE | FOLLOWUP | HALL_INFO | PREFERENCE_RECOMMEND | PREFERENCE_QUERY | LIKE_QUERY | GENERAL",
  "location": "역/지역 이름 또는 null",
  "max_hall_price": "최대 홀 전체 대관료 숫자(원) 또는 null",
  "min_hall_price": "최소 홀 전체 대관료 숫자(원) 또는 null",
  "features": ["주차", "지하철", "버진로드", "야외" 등 시설 조건 배열],
  "mood": ["럭셔리", "로맨틱", "모던" 등 분위기 배열],
  "sort_by": "distance | price_asc | price_desc | rating | review | null",
  "compare_halls": ["비교할 홀 이름1", "홀 이름2"],
  "is_followup": true 또는 false,
  "followup_filter": "이전 결과에서 필터/정렬할 조건 설명 또는 null"
}}

intent 선택 기준:
- HALL_INFO: 특정 홀 1개의 상세정보 조회 (대관료, 식대, 수용인원, 주소, 주차, 예식형태 등)
- LOCATION: 특정 역/위치 근처 검색 (근처, 가까운, 옆에, 역에서 등)
- NEO4J_RAG: 특성/스타일/분위기/가격으로 웨딩홀 검색
- COMPARE: 두 개 이상 홀 비교
- FOLLOWUP: 이전 추천 결과에서 추가 조건으로 필터 (이중에서, 그중에서, 주차되는곳, 더 싼곳 등)
- PREFERENCE_RECOMMEND: 내 취향/선호 기반 추천
- PREFERENCE_QUERY: 내 선호 설정 조회
- LIKE_QUERY: 찜 목록 조회
- GENERAL: 그 외

주의: 이전 추천 결과에 추가 조건을 묻는 경우(주차 잘되는곳, 이중에서 싼곳 등)는 반드시 FOLLOWUP으로 분류하세요.
"""
        try:
            result = llm.invoke(prompt).content.strip()
            import re as _re
            # JSON 블록 추출
            match = _re.search(r'\{.*\}', result, _re.DOTALL)
            if match:
                parsed = json.loads(match.group())
                return parsed
        except Exception as e:
            print(f"parse_intent error: {e}")

        # fallback: 키워드 기반
        msg = message.strip()
        hall_info_kw = ["대관료", "식대", "수용인원", "보증인원", "주소", "연락처", "전화", "예식형태", "식사형태"]
        if any(x in msg for x in hall_info_kw):
            return {"intent": "HALL_INFO", "is_followup": False, "features": [], "mood": []}
        if any(x in msg for x in ["근처", "가까운", "옆에", "이내"]):
            return {"intent": "LOCATION", "location": None, "features": [], "mood": [],
                    "max_hall_price": None, "sort_by": "distance", "is_followup": False}
        if any(x in msg for x in ["비교"]):
            return {"intent": "COMPARE", "compare_halls": [], "is_followup": False}
        return {"intent": "NEO4J_RAG", "is_followup": False, "features": [], "mood": []}

    def classify_intent(message: str, chat_history=None) -> str:
        """하위 호환용 - parse_intent의 intent 필드만 반환"""
        return parse_intent(message, chat_history).get("intent", "GENERAL")

    def default_llm(message: str, chat_history=None) -> str:
        history_text = format_history_for_prompt(chat_history) if chat_history else ""
        history_section = f"\n[이전 대화]\n{history_text}\n" if history_text else ""

        prompt_text = f"""당신은 웨딩홀 추천 전문 챗봇입니다. 친절하고 전문적으로 답변해주세요.
{history_section}
[답변 가이드라인]
1. 사용자가 특정 홀, 트렌드, 비교하기를 물어볼 경우 구체적으로 알려주세요.
2. 가격대, 스타일, 위치, 리뷰, 특색 도 함께 알려주세요.
3. 답변 이후 자연스럽게 추가 질문을 해주세요.

user_input: {message}
"""
        return llm.invoke(prompt_text).content

    def response_fn(message, chat_history):
        chat_history = chat_history or []
        parsed = parse_intent(message, chat_history)
        intent = parsed.get("intent", "GENERAL")
        p_location = parsed.get("location")
        p_max_price = parsed.get("max_hall_price")
        p_min_price = parsed.get("min_hall_price")
        p_features = parsed.get("features") or []
        p_mood = parsed.get("mood") or []
        p_sort_by = parsed.get("sort_by")
        p_compare_halls = parsed.get("compare_halls") or []
        p_is_followup = parsed.get("is_followup", False)
        p_followup_filter = parsed.get("followup_filter")

        # 이전 대화를 RAG/LLM 컨텍스트에 포함할 쿼리 생성
        history_text = format_history_for_prompt(chat_history, max_turns=3)

        # 사용자 선호 정보 주입
        user_pref = get_user_preference(conn, COUPLE_ID)
        pref_region = None
        if user_pref:
            pref_region = user_pref.get("venue") or user_pref.get("region")
        pref_hint = f"[사용자 선호 지역: {pref_region}]\n" if pref_region else ""

        # DB에서 이전 대화 기록 불러오기 (세션 간 지속)
        db_history_text = build_context_from_db_history(COUPLE_ID, max_turns=6)

        # DB 기록과 현재 세션 기록 합치기 (현재 세션 우선)
        combined_history = db_history_text if not history_text else history_text

        if combined_history:
            context_query = f"{pref_hint}이전 대화:\n{combined_history}\n\n현재 질문: {message}"
        else:
            context_query = f"{pref_hint}{message}"

        if intent == "FOLLOWUP":
            # FOLLOWUP이지만 위치 검색이면 이전 결과를 기준으로 재정렬
            if p_location and p_location not in ["null", ""]:
                # 이전 홀들 기준 새 위치와 거리 비교
                prev_halls_raw = extract_hall_names_from_history(chat_history)
                if prev_halls_raw:
                    resolved = resolve_multiple_hall_names(driver, prev_halls_raw)
                    if resolved:
                        rows = compare_multiple_halls(driver, resolved)
                        filter_prompt = f"""이전에 추천한 웨딩홀들: {json.dumps(rows, ensure_ascii=False)}
사용자 질문: {message}
조건: {p_followup_filter or message}

위 실제 데이터만 사용해서 답변하세요. 없는 정보는 만들지 마세요."""
                        answer = llm.invoke(filter_prompt).content
                        cypher = "FOLLOWUP - location rerank"
                        items = safe_str(rows)
                        chat_history.append({"role": "user", "content": message})
                        chat_history.append({"role": "assistant", "content": answer})
                        save_message_to_db(COUPLE_ID, "user", message, intent)
                        save_message_to_db(COUPLE_ID, "assistant", answer, intent)
                        return chat_history, cypher, items

            # mood/feature 조건만 있고 이전 결과 대상 재필터
            if (p_mood or p_features) and not p_location:
                prev_names = extract_hall_names_from_history(chat_history)
                if prev_names:
                    # StyleFilter + memoContent 기반으로 이전 홀들 재평가
                    try:
                        rescore_cypher = """
                        MATCH (h:Hall)
                        WHERE h.name IN $names
                        OPTIONAL MATCH (h)-[:HAS_STYLE_FILTER]->(sf:StyleFilter)
                        WITH h, collect(sf.name) AS styles
                        RETURN h.partnerId AS hallId, h.name AS name, h.rating AS rating,
                               h.reviewCnt AS reviewCnt, h.region AS region,
                               h.minIndividualHallPrice AS minHallPrice,
                               h.memoContent AS memoContent, styles
                        """
                        with driver.session() as dsession:
                            rs = dsession.run(rescore_cypher, names=prev_names)
                            prev_data = [dict(r) for r in rs]

                        filter_prompt = f"""이전에 추천한 웨딩홀 목록과 데이터:
{json.dumps(prev_data, ensure_ascii=False, indent=2)}

사용자 질문: {message}
요청 조건: 분위기={p_mood}, 시설={p_features}

위 데이터에서 조건에 맞는 홀을 골라 답변하세요.
StyleFilter(styles 필드)와 memoContent를 보고 판단하세요.
조건에 맞는 홀이 없으면 솔직하게 알리고 대안을 제안하세요.
데이터에 없는 정보는 만들지 마세요."""
                        answer = llm.invoke(filter_prompt).content
                        cypher = "FOLLOWUP - mood/feature refilter"
                        items = safe_str(prev_data)
                        chat_history.append({"role": "user", "content": message})
                        chat_history.append({"role": "assistant", "content": answer})
                        save_message_to_db(COUPLE_ID, "user", message, intent)
                        save_message_to_db(COUPLE_ID, "assistant", answer, intent)
                        return chat_history, cypher, items
                    except Exception as e:
                        print(f"FOLLOWUP refilter error: {e}")

            hall_names = extract_hall_names_from_history(chat_history)

            if not hall_names:
                # 홀 이름 추출 실패 → 이전 맥락 포함해서 LLM으로 응답
                answer = default_llm(message, chat_history)
                cypher = "FOLLOWUP - hall name extraction failed"
                items = ""
            else:
                # Neo4j에서 해당 홀 조회
                cypher_query = f"""
MATCH (h:Hall)
WHERE h.name IN {json.dumps(hall_names, ensure_ascii=False)}
OPTIONAL MATCH (h)-[:IN_DISTRICT]->(d:District)
OPTIONAL MATCH (h)-[:HAS_STYLE_FILTER]->(sf:StyleFilter)
WITH h, d, collect(sf.name) AS styles
RETURN h.name AS name, h.rating AS rating, h.reviewCnt AS reviewCnt,
       h.address AS address, h.subRegion AS district,
       h.memoContent AS memoContent,
       h.minIndividualHallPrice AS minHallPrice,
       h.minMealPrice AS minMealPrice,
       styles
ORDER BY h.rating DESC
"""
                with driver.session() as session:
                    result = session.run(cypher_query)
                    rows = [dict(r) for r in result]

                halls_str = ", ".join(hall_names)

                if not rows:
                    # DB에 없는 홀들 → 솔직하게 고지하고 유사 홀 검색 제안
                    not_found_prompt = f"""이전에 추천한 웨딩홀 목록: {halls_str}

사용자 질문: {message}

위 홀들은 우리 데이터베이스에 등록되어 있지 않아 정확한 정보(평점, 위치 등)를 제공하기 어렵습니다.
사용자에게 이를 솔직히 알리고, 대신 우리 DB에서 비슷한 스타일의 웨딩홀을 찾아볼 것을 제안하세요.
답변은 친절하게, 2~3문장으로 간결하게 작성하세요.
"""
                    answer = llm.invoke(not_found_prompt).content
                    cypher = cypher_query.strip()
                    items = safe_str({"searched_names": hall_names, "found": []})
                else:
                    # DB에 있는 홀들 → LLM이 조건에 맞게 필터/정렬해서 답변
                    filter_prompt = f"""이전에 추천한 웨딩홀 목록: {halls_str}

사용자 질문: {message}

아래는 우리 데이터베이스에서 조회한 실제 정보입니다:
{json.dumps(rows, ensure_ascii=False, indent=2)}

위 실제 데이터를 바탕으로 사용자 질문에 답변하세요.
- 데이터에 없는 정보는 지어내지 마세요.
- 조건에 맞는 홀이 없으면 솔직하게 말하세요.
"""
                    answer = llm.invoke(filter_prompt).content
                    cypher = cypher_query.strip()
                    items = safe_str(rows)

        elif intent == "PREFERENCE_QUERY":
            user_pref = get_user_preference(conn, COUPLE_ID)
            if user_pref is None:
                answer = "저장된 선호 정보가 없습니다."
                items = ""
            else:
                answer = (
                    f"현재 설정된 선호 정보입니다.\n"
                    f"- 지역: {user_pref.get('venue', user_pref.get('region', '미설정'))}\n"
                    f"- 스타일: {user_pref.get('style', '미설정')}\n"
                    f"- 분위기: {user_pref.get('mood', '미설정')}\n"
                    f"- 식사: {user_pref.get('food', '미설정')}\n"
                    f"- 예산: {user_pref.get('budget', '미설정')}"
                )
                items = safe_str(user_pref)
            cypher = "MySQL - couple_preferences"

        elif intent == "PREFERENCE_RECOMMEND":
            user_pref = get_user_preference(conn, COUPLE_ID)
            if user_pref is None:
                answer = "저장된 선호 정보가 없어 기본 추천을 드리기 어렵습니다."
                items = ""
                cypher = "couple_preferences 없음"
            else:
                halls = recommend_by_preference(driver, user_pref)
                if not halls:
                    answer = "현재 선호 조건에 맞는 웨딩홀을 찾지 못했습니다."
                    items = ""
                else:
                    lines = [f"- {h['name']} (평점 {h['rating']}, 리뷰 {h['reviewCnt']}, 추천점수 {h['score']})" for h in halls[:5]]
                    answer = "선호 조건에 맞춰서 추천해드린 웨딩홀입니다.\n\n" + "\n".join(lines)
                    items = safe_str(halls)
                cypher = "Preference-based Cypher"

        elif intent == "LIKE_QUERY":
            likes = get_user_likes(conn, COUPLE_ID)
            if not likes:
                answer = "찜하신 웨딩홀이 없습니다."
                items = ""
            else:
                answer = f"찜하신 웨딩홀 목록이 {len(likes)}개 있습니다."
                items = safe_str(likes)
            cypher = "MySQL - couple_venue_likes"

        elif intent == "LIKE_BASED_RECOMMEND":
            user_pref = get_user_preference(conn, COUPLE_ID)
            preferred_region = None
            if user_pref:
                preferred_region = user_pref.get("region") or user_pref.get("venue")
            rec = recommend_by_likes_with_region_boost(conn, COUPLE_ID, preferred_region=preferred_region)
            if not rec:
                answer = "찜한 기록이 없어 협업 필터링 추천이 어렵습니다."
                items = ""
            else:
                lines = [f"- {r['name']} (평점 {r['rating']})" for r in rec]
                answer = "찜 기록과 선호 지역을 반영한 추천 결과입니다.\n\n" + "\n".join(lines)
                items = safe_str(rec)
            cypher = "MySQL Collaborative Filtering + Region"

        elif intent == "FACILITY_SEARCH":
            keyword = extract_facility_keyword(message)
            region = extract_region(message)
            if region is None:
                user_pref = get_user_preference(conn, COUPLE_ID)
                if user_pref:
                    region = user_pref.get("region") or user_pref.get("venue")
            halls = recommend_by_facility(driver, keyword, region)
            if not halls:
                answer = f"'{keyword}' 시설 웨딩홀을 찾지 못했습니다."
                items = ""
            else:
                lines = [f"- {h['name']} (평점 {h['rating']}, 리뷰 {h['reviewCnt']})" for h in halls[:5]]
                prefix = f"{region} 지역에서 " if region else ""
                answer = f"{prefix}'{keyword}' 시설을 갖춘 추천 웨딩홀입니다.\n\n" + "\n".join(lines)
                items = safe_str(halls)
            cypher = f"Facility Search: {keyword}, region={region}"

        elif intent == "HALL_INFO":
            # 특정 홀 정보 직접 Neo4j 쿼리
            # 홀 이름 추출 (LLM)
            name_prompt = f"""다음 메시지에서 웨딩홀 이름만 추출하세요. JSON 배열로만 반환하세요.
메시지: {message}
출력 예시: ["엘타워"]"""
            try:
                name_result = llm.invoke(name_prompt).content.strip()
                import re as _re
                name_match = _re.search(r'\[.*?\]', name_result, _re.DOTALL)
                hall_keywords = json.loads(name_match.group()) if name_match else []
            except Exception:
                hall_keywords = []

            if not hall_keywords:
                # 이름 추출 실패 → RAG로 fallback
                rag_result = rag.search(query_text=context_query, return_context=True)
                answer = rag_result.answer
                cypher = rag_result.retriever_result.metadata.get("cypher", "")
                items = safe_str(rag_result.retriever_result.items)
            else:
                # 직접 Neo4j 쿼리
                info_cypher = """
MATCH (h:Hall)
WHERE any(kw IN $keywords WHERE h.name CONTAINS kw)
OPTIONAL MATCH (h)-[:HAS_STYLE_FILTER]->(sf:StyleFilter)
WITH h, collect(sf.name) AS styles
RETURN h.name AS name,
       h.minIndividualHallPrice AS minHallPrice,
       h.maxIndividualHallPrice AS maxHallPrice,
       h.minRentalPrice AS minRentalPrice,
       h.maxRentalPrice AS maxRentalPrice,
       h.minMealPrice AS minMealPrice,
       h.maxMealPrice AS maxMealPrice,
       h.address AS address,
       h.region AS region,
       h.subRegion AS district,
       h.tel AS tel,
       h.rating AS rating,
       h.reviewCnt AS reviewCnt,
       h.memoContent AS memoContent,
       styles
ORDER BY h.reviewCnt DESC
LIMIT 3
"""
                with driver.session() as session:
                    result = session.run(info_cypher, keywords=hall_keywords)
                    rows = [dict(r) for r in result]

                if not rows:
                    answer = f"'{', '.join(hall_keywords)}'에 해당하는 웨딩홀을 찾지 못했습니다."
                    items = ""
                else:
                    info_prompt = f"""사용자 질문: {message}

웨딩홀 데이터:
{json.dumps(rows, ensure_ascii=False, indent=2)}

위 데이터를 바탕으로 사용자 질문에 친절하게 답변하세요.
- 가격은 원 단위로 콤마 구분해서 표시 (예: 51,500,000원)
- memoContent에서 관련 정보(수용인원, 교통, 주차 등)를 찾아서 답변
- 데이터에 없는 정보는 만들지 마세요"""
                    answer = llm.invoke(info_prompt).content
                    items = safe_str(rows)

                cypher = info_cypher.strip()

        elif intent == "COMPARE":
            hall_keywords = extract_multiple_halls(message)
            resolved_names = resolve_multiple_hall_names(driver, hall_keywords)

            # 현재 메시지에서 못 찾으면 이전 대화에서 홀 이름 추출
            if len(resolved_names) < 2:
                history_hall_names = extract_hall_names_from_history(chat_history)
                if len(history_hall_names) >= 2:
                    resolved_names = resolve_multiple_hall_names(driver, history_hall_names)

            if len(resolved_names) < 2:
                answer = "비교할 웨딩홀 이름을 두 개 이상 찾지 못했습니다. 비교할 홀 이름을 직접 말씀해 주세요."
                cypher = "COMPARE resolve failed"
                items = safe_str({"input_keywords": hall_keywords, "resolved_names": resolved_names})
            else:
                rows = compare_multiple_halls(driver, resolved_names)
                answer = make_multi_compare_answer(rows) if len(rows) >= 2 else "웨딩홀 상세 정보를 찾지 못했습니다."
                items = safe_str(rows)
                cypher = "MULTI HALL COMPARE QUERY"

        elif intent == "LOCATION":
            loc_msg = message
            if p_location:
                loc_msg = p_location + " 근처 웨딩홀"
            result = recommend_by_location(driver, conn, COUPLE_ID, loc_msg)
            if result.get("error"):
                answer = result["error"]
                items = ""
                cypher = "Location search failed"
            else:
                halls = result["results"]
                start_location = result["start_location"]
                region = result["region"]

                # parse_intent에서 뽑은 조건으로 필터
                if p_max_price:
                    halls = [h for h in halls if (h.get("minHallPrice") or 999999999) <= p_max_price]
                if p_min_price:
                    halls = [h for h in halls if (h.get("minHallPrice") or 0) >= p_min_price]
                if p_features:
                    def has_feature(h, feats):
                        memo = (h.get("memoContent") or "") + (h.get("name") or "")
                        return all(f in memo for f in feats)
                    halls = [h for h in halls if has_feature(h, p_features)]
                if p_mood:
                    # StyleFilter 노드에서 mood와 일치하는 홀 ID 조회
                    mood_hall_ids = set()
                    try:
                        mood_cypher = """
                        MATCH (h:Hall)-[:HAS_STYLE_FILTER]->(sf:StyleFilter)
                        WHERE any(m IN $moods WHERE sf.name CONTAINS m)
                        RETURN h.partnerId AS hallId
                        """
                        with driver.session() as dsession:
                            mood_result = dsession.run(mood_cypher, moods=p_mood)
                            mood_hall_ids = {r["hallId"] for r in mood_result}
                    except Exception as e:
                        print(f"StyleFilter query error: {e}")

                    # StyleFilter 매칭 홀 우선 정렬 + memoContent도 체크
                    def mood_score(h):
                        memo = (h.get("memoContent") or "") + (h.get("name") or "")
                        sf_match = 1 if h.get("hallId") in mood_hall_ids else 0
                        memo_match = sum(1 for m in p_mood if m in memo)
                        return -(sf_match * 3 + memo_match)

                    halls = sorted(halls, key=lambda h: (mood_score(h), h.get("distance_km", 99)))

                # 정렬
                sort_by = p_sort_by or "distance"
                if sort_by == "price_desc":
                    halls = sorted(halls, key=lambda x: -(x.get("minHallPrice") or 0))
                elif sort_by == "price_asc":
                    halls = sorted(halls, key=lambda x: (x.get("minHallPrice") or 99999999))
                elif sort_by == "rating":
                    halls = sorted(halls, key=lambda x: -(x.get("rating") or 0))
                elif sort_by == "review":
                    halls = sorted(halls, key=lambda x: -(x.get("reviewCnt") or 0))

                if not halls:
                    cond_str = []
                    if p_max_price: cond_str.append(f"{p_max_price:,}원 이하")
                    if p_features: cond_str.append("/".join(p_features))
                    cond_desc = " + ".join(cond_str) if cond_str else "해당 조건"
                    answer = f"{start_location} 근처에서 {cond_desc}에 맞는 웨딩홀을 찾지 못했습니다."
                    items = ""
                else:
                    top5 = halls[:5]
                    sort_label = {"distance":"가까운 순","price_desc":"가격 높은 순",
                                  "price_asc":"가격 낮은 순","rating":"평점 높은 순","review":"리뷰 많은 순"}.get(sort_by,"가까운 순")
                    cond_parts = []
                    if p_max_price: cond_parts.append(f"{p_max_price:,}원 이하")
                    if p_features: cond_parts.append("/".join(p_features) + " 있는 곳")
                    if p_mood: cond_parts.append("/".join(p_mood) + " 분위기")
                    cond_desc = ", ".join(cond_parts)

                    lines = []
                    for h in top5:
                        price_info = f", 홀대관료 {h['minHallPrice']:,}원~" if h.get("minHallPrice") else ""
                        lines.append(f"- {h['name']} ({h.get('address','주소미상')}{price_info})")

                    header = f"{start_location} 기준 {sort_label}"
                    if cond_desc: header += f" / {cond_desc}"
                    answer = header + " 웨딩홀입니다.\n\n" + "\n".join(lines)
                    items = safe_str(halls)
                cypher = result.get("neo4j_query", f"Location start={start_location}, region={region}")

        elif intent == "NEO4J_RAG":
            rag_result = rag.search(query_text=context_query, return_context=True)
            cypher = rag_result.retriever_result.metadata.get("cypher", "")
            items = rag_result.retriever_result.items

            # DB 조회 결과가 없으면 hallucination 방지
            if not items:
                answer = "저희 데이터베이스에서 해당 조건에 맞는 웨딩홀을 찾지 못했습니다. 다른 조건으로 검색해 보시겠어요?"
            else:
                answer = rag_result.answer

            items = safe_str(items)

        else:
            answer = default_llm(message, chat_history)
            cypher = "일반 대화"
            items = "일반 대화"

        chat_history.append({"role": "user", "content": message})
        chat_history.append({"role": "assistant", "content": answer})

        # MySQL에 대화 저장
        save_message_to_db(COUPLE_ID, "user", message, intent)
        save_message_to_db(COUPLE_ID, "assistant", answer, intent)

        return chat_history, cypher, items

    with gr.Row():
        gr.HTML("""
        <div style="text-align:center; max-width:1000px; margin:10px auto;">
            <h1>Graph RAG 웨딩홀 챗봇</h1>
            <p>Neo4j 그래프 DB 기반으로 웨딩홀을 추천하고, 생성된 Cypher와 조회 결과를 함께 보여드립니다.</p>
        </div>
        """)

    with gr.Row():
        with gr.Column(scale=1):
            generated_query = gr.Textbox(label="생성된 Cypher 쿼리", lines=18)
            query_result = gr.Textbox(label="원본 조회 결과", lines=18)

        with gr.Column(scale=3):
            chatbot = gr.Chatbot(type="messages")
            msg = gr.Textbox(
                placeholder="예: 강남 근처에서 접근성 좋은 웨딩홀 추천해줘",
                label="입력"
            )

            gr.Examples(
                examples=[
                    "웨딩홀 그랜드볼룸 자연채광이 있는 모던한 분위기의 웨딩홀을 추천해줄 수 있나요?",
                    "웨딩홀 라온파라디아회화나무 혹은 채플 스타일 웨딩홀을 추천해주세요.",
                    "서울 강남지역에서 접근성 좋고 리뷰가 많은 웨딩홀 5개 추천해줘",
                    "내 취향에 맞는 웨딩홀 알려줘",
                    "현재 찜한 웨딩홀 목록",
                    "찜한 기반 웨딩홀을 찾아 줘 추천해줘",
                    "강남역에서 가까운 웨딩홀 추천해줘"
                ],
                inputs=[msg]
            )

            with gr.Row():
                btn = gr.Button("Submit", variant="primary")
                clear = gr.Button("Clear")

    submit_event = dict(
        fn=response_fn,
        inputs=[msg, chatbot],
        outputs=[chatbot, generated_query, query_result]
    )

    btn.click(**submit_event).then(lambda: "", None, msg)
    msg.submit(**submit_event).then(lambda: "", None, msg)

    def clear_all():
        return [], "", ""

    clear.click(
        fn=clear_all,
        inputs=None,
        outputs=[chatbot, generated_query, query_result],
        queue=False
    ).then(lambda: "", None, msg)

demo.launch(debug=True, share=True)


C:\Users\SSAFY\AppData\Local\Temp\ipykernel_26648\1368167436.py:1: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=seafoam) as demo:
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_26648\1368167436.py:265: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(type="messages", )


* Running on local URL:  http://127.0.0.1:7864

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026/03/16 09:47:07 [W] [service.go:132] login to server failed: dial tcp 44.237.78.176:7000: i/o timeout
